# CXR-Recall — Phase 1 multi-seed run (Arm 1 vs Arm 3c-frozen)

Separate notebook from `cxr_recall_arm1_kaggle.ipynb`. It exists because the 2026-09-06 audit rerun showed two things (RESULTS.md, "Phase 1 audit rerun"): `SEED=42` does **not** reproduce Arm 1 (0.7092 in one run, 0.7272 in the next, on identical code — GPU kernel non-determinism), and the H1 margin (Arm 3c-frozen − Arm 1 = +0.0030 val / +0.0108 test) sits inside Arm 1's ~0.024 run-to-run range. A single seed therefore cannot establish a magnitude for H1. This notebook trains **Arm 1 and Arm 3c-frozen under several seeds** and reports the margin as a distribution (mean ± std, 95% CI, sign count, paired t-test), on the held-out **test** split.

**What varies per seed and what doesn't.** The patient-level split is fixed at `seed=42` (`patient_level_split`'s own argument, independent of `set_seed`), so every seed is scored on the *same* 743 test images and the per-seed margins are paired. Only model initialisation and DataLoader shuffle order change with the seed. Arm 3c-frozen always takes *that seed's* Arm 1 encoder, byte-identical and frozen, so each margin is a clean within-seed comparison.

**Cost and session limits.** Per seed: Arm 1 ≈ 47 min, embedding extraction ≈ 5 min, Arm 3c-frozen ≈ 48 min → ≈ 1.7 h on a Kaggle T4. The default `SEEDS = [0, 1, 2]` (≈ 5 h) stays well inside the 12 h session cap. Run a second session with `SEEDS = [3, 4]` and paste the first session's per-seed `RESULT_JSON` lines into `PRIOR_RESULTS` so the final cell aggregates all of them. Unfrozen Arm 3c is deliberately not trained — it is the frozen variant's margin that the project's H1 claim rests on.

Attach the same dataset (+ Add Data → "Chest X-rays Indiana University"), GPU on. Use **Save Version → Save & Run All** so a timeout can't kill an interactive session mid-seed.

In [ ]:
import torch
print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

In [ ]:
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# NOTE: this pins init + shuffle order per seed, NOT GPU kernel non-determinism -- the
# audit rerun showed the same seed can still land 0.018 apart on Arm 1. That is exactly
# why this notebook reports a distribution over seeds instead of one number.
# The data split below is a separate, fixed seed=42 (patient_level_split's own argument).
set_seed(42)  # for the cheap data-prep cells; each training block re-seeds with its own seed
print('set_seed defined')

## 1. Locate the attached dataset

In [ ]:
import os

INPUT_DIR = '/kaggle/input'
candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached — use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 2. Build the manifest (identical to the Phase 1 notebook, Section 2)

In [ ]:
reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

## 3. Label vocabulary + fixed patient-level split (`seed=42`, identical to Phase 1)

In [ ]:
from collections import Counter
import numpy as np

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

In [ ]:
def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

## 4. Dataset, Arm 1 model, loaders, metrics (identical to Phase 1, Sections 5-6)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class IUXrayDataset(Dataset):
    def __init__(self, df, image_root, transform, label_vocab):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
        self.label_vocab = label_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        image = self.transform(image)
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        label = mesh_tags_to_vector(tags, self.label_vocab)
        return image, label

In [ ]:
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

class NoRetrievalClassifier(nn.Module):
    def __init__(self, num_labels, pretrained=True):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.head = nn.Linear(768, num_labels)

    def forward(self, images):
        return self.head(self.backbone(images))

transform = ViT_B_16_Weights.DEFAULT.transforms()
train_ds = IUXrayDataset(train_df, IMAGE_ROOT, transform, label_vocab)
val_ds = IUXrayDataset(val_df, IMAGE_ROOT, transform, label_vocab)
test_ds = IUXrayDataset(test_df, IMAGE_ROOT, transform, label_vocab)  # held out: never used for early stopping or model selection
print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR = 16, 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

criterion = nn.BCEWithLogitsLoss()

def macro_auroc(y_true, y_score):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(scores)) if scores else float('nan')

def held_out_auroc(forward_fn, loader):
    """One pass over a held-out loader with the model already in eval() mode.
    forward_fn(batch_tuple) -> logits. Every loader in this notebook yields
    labels at position 1. Used for the TEST numbers: val is what every arm's
    early stopping selects on, so val AUROC is an optimistically-selected
    number; test is evaluated exactly once per arm, on the reloaded best
    checkpoint, and never influences any training or selection decision."""
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            all_scores.append(torch.sigmoid(forward_fn(batch)).cpu().numpy())
            all_labels.append(batch[1].numpy())
    return macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))

print('setup complete -- device:', device, ' batch size:', BATCH_SIZE, ' lr:', LR)

## 5. Retrieval utilities + evidence dataset + Arm 3c model (identical to Phase 1, Sections 7-9)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

@torch.no_grad()
def extract_embeddings(backbone, loader, device):
    backbone.eval()
    all_emb, all_labels = [], []
    for images, labels in loader:
        emb = backbone(images.to(device)).cpu().numpy()
        all_emb.append(emb)
        all_labels.append(labels.numpy())
    return np.concatenate(all_emb), np.concatenate(all_labels)

In [ ]:
def build_index(embeddings):
    embeddings = embeddings.astype(np.float32).copy()
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def search(index, queries, k):
    queries = queries.astype(np.float32).copy()
    faiss.normalize_L2(queries)
    return index.search(queries, k)

def label_agreement_at_k(query_labels, retrieved_labels):
    if retrieved_labels.shape[0] == 0:
        return float('nan')
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    return float(shares.mean())

def mean_reciprocal_rank(query_labels, retrieved_labels):
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    hits = np.nonzero(shares)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)

K = 5
print('retrieval utilities defined -- k =', K)

In [ ]:
class EvidenceAugmentedDatasetV2(Dataset):
    """Wraps IUXrayDataset, additionally returning each item's k retrieved
    neighbors' label vectors AND image embeddings -- feeds Arm 3c below.
    (The 'V2' name is kept only because Section 11's cells already reference
    it; Arm 3b, which originally introduced it, is no longer built here.)"""
    def __init__(self, base_df, image_root, transform, label_vocab, evidence_pool_labels, evidence_pool_embeddings, neighbor_idx):
        self.base = IUXrayDataset(base_df, image_root, transform, label_vocab)
        self.evidence_pool_labels = evidence_pool_labels
        self.evidence_pool_embeddings = evidence_pool_embeddings
        self.neighbor_idx = neighbor_idx

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, label = self.base[idx]
        idxs = self.neighbor_idx[idx]
        evidence_labels = self.evidence_pool_labels[idxs]
        evidence_images = self.evidence_pool_embeddings[idxs]
        return image, label, torch.tensor(evidence_labels, dtype=torch.float32), torch.tensor(evidence_images, dtype=torch.float32)

In [ ]:
class EvidenceEncoderV3(nn.Module):
    """Joint attention: image and label tokens stay separate (2k total),
    tagged with a modality embedding and a neighbor-position embedding."""
    def __init__(self, num_labels, embed_dim, num_heads=4, image_embed_dim=None, max_k=16):
        super().__init__()
        image_embed_dim = image_embed_dim or embed_dim
        self.label_proj = nn.Linear(num_labels, embed_dim)
        self.image_proj = nn.Linear(image_embed_dim, embed_dim)
        self.modality_embed = nn.Embedding(2, embed_dim)      # 0 = image, 1 = label
        self.neighbor_pos_embed = nn.Embedding(max_k, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, query_embedding, evidence_labels, evidence_image_embeddings):
        k = evidence_labels.shape[1]
        device = evidence_labels.device

        label_embed = self.label_proj(evidence_labels)            # (batch, k, embed_dim)
        image_embed = self.image_proj(evidence_image_embeddings)  # (batch, k, embed_dim)

        pos = self.neighbor_pos_embed(torch.arange(k, device=device)).unsqueeze(0)  # (1, k, embed_dim)
        image_type = self.modality_embed.weight[0]
        label_type = self.modality_embed.weight[1]

        image_tok = image_embed + pos + image_type
        label_tok = label_embed + pos + label_type
        evidence_tokens = torch.cat([image_tok, label_tok], dim=1)  # (batch, 2k, embed_dim) -- separately attendable

        query = query_embedding.unsqueeze(1)
        context, _weights = self.attention(query, evidence_tokens, evidence_tokens)
        return context.squeeze(1)


class RetrievalFusionClassifierV3(nn.Module):
    def __init__(self, num_labels, pretrained=True, image_embed_dim=768, max_k=16):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.evidence_encoder = EvidenceEncoderV3(num_labels, 768, image_embed_dim=image_embed_dim, max_k=max_k)
        self.head = nn.Linear(768, num_labels)

    def forward(self, images, evidence_labels, evidence_image_embeddings, evidence_confidence=None):
        # evidence_confidence: optional (batch,) tensor, hard 0.0/1.0 from a
        # similarity-threshold cutoff (Section 12) -- 0.0 zeroes this sample's
        # evidence contribution entirely, falling back to a pure vision-only
        # prediction for it; omitted (default), behavior is unchanged from
        # every earlier Arm 3c/3c-frozen cell above.
        query_embedding = self.backbone(images)
        evidence_context = self.evidence_encoder(query_embedding, evidence_labels, evidence_image_embeddings)
        if evidence_confidence is not None:
            fused = query_embedding + evidence_context * evidence_confidence.unsqueeze(-1)
        else:
            fused = query_embedding + evidence_context
        return self.head(fused)

## 6. Multi-seed configuration

`SEEDS` is the only thing to edit between sessions. `PRIOR_RESULTS` takes the `RESULT_JSON: {...}` lines printed by an earlier session (copy each `{...}` in as a Python dict) so the aggregate in Section 8 covers every seed run so far.

In [ ]:
import copy, json, time

SEEDS = [0, 1, 2]        # ~1.7 h per seed; [0, 1, 2] ~ 5 h. Second session: SEEDS = [3, 4].
PRIOR_RESULTS = []       # e.g. [{'seed': 0, 'arm1_val': ..., 'arm1_test': ..., ...}, ...] from an earlier session's RESULT_JSON lines
EPOCHS_REAL, PATIENCE = 20, 5
RESULTS_PATH = '/kaggle/working/multiseed_results.json'

# Reference points from the two SEED=42 runs already documented in RESULTS.md (NOT
# included in the statistics below -- they are here so the aggregate cell can show
# where the single-seed numbers fell relative to the distribution).
REFERENCE_RUNS = {
    'seeded run (Phase 1e, val only)':     {'arm1_val': 0.7092, 'arm3cf_val': 0.7353},
    'audit rerun (Phase 1, val / test)':   {'arm1_val': 0.7272, 'arm3cf_val': 0.7302, 'arm1_test': 0.7051, 'arm3cf_test': 0.7159},
}

train_pids = train_df['patient_id'].to_numpy()  # row order == train_ds order (IUXrayDataset resets index in order)
MAX_VIEWS_PER_STUDY = int(train_df.groupby('patient_id').size().max())
train_loader_seq = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('seeds to run this session:', SEEDS, ' prior results supplied:', len(PRIOR_RESULTS))

## 7. Per-seed pipeline

Three functions, each a verbatim copy of the corresponding Phase 1 notebook block (Section 8's Arm 1 loop, Section 8's retrieval/evidence construction with study-level exclusion, Section 10's Arm 3c-frozen loop) with the seed as an argument. The loop at the bottom runs them for each seed, prints one `RESULT_JSON` line per seed the moment that seed finishes, and appends to `/kaggle/working/multiseed_results.json` — so a session that dies mid-way still leaves the completed seeds recoverable from the log.

In [ ]:
def run_arm1(seed):
    """Phase 1 Section 8, Arm 1 -- returns (model with best checkpoint loaded, best_val, best_epoch, test_auroc)."""
    set_seed(seed)
    model = NoRetrievalClassifier(num_labels=len(label_vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_auroc, best_state, best_epoch, epochs_without_improvement = -1.0, None, 0, 0
    for epoch in range(1, EPOCHS_REAL + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        total_loss /= max(len(train_ds), 1)
        model.eval()
        all_labels, all_scores = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                all_scores.append(torch.sigmoid(model(images.to(device))).cpu().numpy())
                all_labels.append(labels.numpy())
        val_auroc = macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))
        print(f'[seed {seed} | Arm 1] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc:.4f}')
        if val_auroc > best_auroc:
            best_auroc, best_state, best_epoch, epochs_without_improvement = val_auroc, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f'[seed {seed} | Arm 1] no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
                break
    model.load_state_dict(best_state)
    model.eval()
    test_auroc = held_out_auroc(lambda b: model(b[0].to(device)), test_loader)
    print(f'[seed {seed} | Arm 1] best val = {best_auroc:.4f} (epoch {best_epoch})   TEST = {test_auroc:.4f}')
    return model, best_auroc, best_epoch, test_auroc


def build_evidence(model, seed):
    """Phase 1 Section 8 retrieval: embeddings from this seed's Arm 1 encoder, Arm 2 metrics on
    val/test, and train-vs-train neighbors with study-level self-exclusion (LEAKAGE CHECK)."""
    train_emb, train_lbl = extract_embeddings(model.backbone, train_loader_seq, device)
    val_emb, val_lbl = extract_embeddings(model.backbone, val_loader, device)
    test_emb, test_lbl = extract_embeddings(model.backbone, test_loader, device)
    index = build_index(train_emb)
    _, val_nb = search(index, val_emb, K)
    _, test_nb = search(index, test_emb, K)
    arm2 = {
        'arm2_val_agreement': float(np.mean([label_agreement_at_k(val_lbl[i], train_lbl[val_nb[i]]) for i in range(len(val_emb))])),
        'arm2_test_agreement': float(np.mean([label_agreement_at_k(test_lbl[i], train_lbl[test_nb[i]]) for i in range(len(test_emb))])),
    }
    _, raw = search(index, train_emb, K + MAX_VIEWS_PER_STUDY)
    train_nb, same_study_hits = [], 0
    for i, row in enumerate(raw):
        index_only = [j for j in row if j != i][:K]
        same_study_hits += int(any(train_pids[j] == train_pids[i] for j in index_only))
        keep = [j for j in row if j != i and train_pids[j] != train_pids[i]][:K]
        assert len(keep) == K
        train_nb.append(keep)
    train_nb = np.array(train_nb)
    print(f'[seed {seed} | Arm 2] label_agreement@{K} val={arm2["arm2_val_agreement"]:.4f} test={arm2["arm2_test_agreement"]:.4f}   '
          f'LEAKAGE CHECK: {same_study_hits}/{len(train_pids)} ({same_study_hits / len(train_pids):.1%})')
    loaders = {
        'train': DataLoader(EvidenceAugmentedDatasetV2(train_df, IMAGE_ROOT, transform, label_vocab, train_lbl, train_emb, train_nb), batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
        'val':   DataLoader(EvidenceAugmentedDatasetV2(val_df, IMAGE_ROOT, transform, label_vocab, train_lbl, train_emb, val_nb), batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
        'test':  DataLoader(EvidenceAugmentedDatasetV2(test_df, IMAGE_ROOT, transform, label_vocab, train_lbl, train_emb, test_nb), batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
    }
    return loaders, arm2, same_study_hits / len(train_pids), train_emb.shape[1]


def run_arm3c_frozen(seed, backbone_state, loaders, image_embed_dim):
    """Phase 1 Section 10, Arm 3c-frozen -- encoder loaded from THIS seed's Arm 1 and frozen."""
    set_seed(seed)
    m = RetrievalFusionClassifierV3(num_labels=len(label_vocab), image_embed_dim=image_embed_dim, max_k=K).to(device)
    m.backbone.load_state_dict(backbone_state)
    for p in m.backbone.parameters():
        p.requires_grad = False
    m.backbone.eval()
    optimizer = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=LR)
    best_auroc, best_state, best_epoch, epochs_without_improvement = -1.0, None, 0, 0
    for epoch in range(1, EPOCHS_REAL + 1):
        m.train(); m.backbone.eval()
        total_loss = 0.0
        for images, labels, ev_labels, ev_images in loaders['train']:
            images, labels = images.to(device), labels.to(device)
            ev_labels, ev_images = ev_labels.to(device), ev_images.to(device)
            optimizer.zero_grad()
            loss = criterion(m(images, ev_labels, ev_images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        total_loss /= max(len(loaders['train'].dataset), 1)
        m.eval()
        all_labels, all_scores = [], []
        with torch.no_grad():
            for images, labels, ev_labels, ev_images in loaders['val']:
                all_scores.append(torch.sigmoid(m(images.to(device), ev_labels.to(device), ev_images.to(device))).cpu().numpy())
                all_labels.append(labels.numpy())
        val_auroc = macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))
        print(f'[seed {seed} | Arm 3c-frozen] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc:.4f}')
        if val_auroc > best_auroc:
            best_auroc, best_state, best_epoch, epochs_without_improvement = val_auroc, copy.deepcopy(m.state_dict()), epoch, 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f'[seed {seed} | Arm 3c-frozen] no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
                break
    m.load_state_dict(best_state)
    m.eval()
    test_auroc = held_out_auroc(lambda b: m(b[0].to(device), b[2].to(device), b[3].to(device)), loaders['test'])
    # evidence-ablation shift on val (same measurement as every prior run)
    real, zero_all, zero_img = [], [], []
    with torch.no_grad():
        for images, labels, ev_labels, ev_images in loaders['val']:
            images_d, ev_labels_d, ev_images_d = images.to(device), ev_labels.to(device), ev_images.to(device)
            real.append(torch.sigmoid(m(images_d, ev_labels_d, ev_images_d)).cpu().numpy())
            zero_all.append(torch.sigmoid(m(images_d, torch.zeros_like(ev_labels_d), torch.zeros_like(ev_images_d))).cpu().numpy())
            zero_img.append(torch.sigmoid(m(images_d, ev_labels_d, torch.zeros_like(ev_images_d))).cpu().numpy())
    real, zero_all, zero_img = map(np.concatenate, (real, zero_all, zero_img))
    shift_full, shift_img = float(np.mean(np.abs(real - zero_all))), float(np.mean(np.abs(real - zero_img)))
    print(f'[seed {seed} | Arm 3c-frozen] best val = {best_auroc:.4f} (epoch {best_epoch})   TEST = {test_auroc:.4f}   shift full/img = {shift_full:.4f}/{shift_img:.4f}')
    return best_auroc, best_epoch, test_auroc, shift_full, shift_img


print('per-seed functions defined')

In [ ]:
results = []
for seed in SEEDS:
    t0 = time.time()
    print(f'\n================ SEED {seed} ================')
    model, arm1_val, arm1_epoch, arm1_test = run_arm1(seed)
    loaders, arm2, leakage_rate, image_embed_dim = build_evidence(model, seed)
    backbone_state = copy.deepcopy(model.backbone.state_dict())
    del model; torch.cuda.empty_cache()
    arm3cf_val, arm3cf_epoch, arm3cf_test, shift_full, shift_img = run_arm3c_frozen(seed, backbone_state, loaders, image_embed_dim)
    del backbone_state, loaders; torch.cuda.empty_cache()

    r = {'seed': seed,
         'arm1_val': round(arm1_val, 4), 'arm1_test': round(arm1_test, 4), 'arm1_epoch': arm1_epoch,
         'arm3cf_val': round(arm3cf_val, 4), 'arm3cf_test': round(arm3cf_test, 4), 'arm3cf_epoch': arm3cf_epoch,
         # margins from the rounded arm values so the table is self-consistent to the last digit
         'margin_val': round(round(arm3cf_val, 4) - round(arm1_val, 4), 4), 'margin_test': round(round(arm3cf_test, 4) - round(arm1_test, 4), 4),
         'shift_full': round(shift_full, 4), 'shift_img': round(shift_img, 4),
         'leakage_rate': round(leakage_rate, 4), **{k: round(v, 4) for k, v in arm2.items()},
         'minutes': round((time.time() - t0) / 60, 1)}
    results.append(r)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=1)
    print(f'RESULT_JSON: {json.dumps(r)}')   # copy this line into PRIOR_RESULTS for a later session
print(f'\nsession complete: {len(results)} seeds in {sum(r["minutes"] for r in results):.0f} min')

## 8. Aggregate across seeds

Test-set margins are paired (same 743 images, same seed for both arms), so the right test is a one-sample t-test of the per-seed margin against zero. With 3–5 seeds the CI will be wide — that is the honest width, and it is the number to report instead of any single run's margin.

In [ ]:
from scipy import stats

all_results = sorted(PRIOR_RESULTS + results, key=lambda r: r['seed'])
n = len(all_results)
print(f'=== Multi-seed Phase 1: Arm 1 vs Arm 3c-frozen, {n} seed(s) ===')
print('seed | Arm1 val  test  | Arm3cf val  test  | margin val    TEST   | shift full/img | Arm2 val/test | leak | min')
for r in all_results:
    print(f"{r['seed']:>4} | {r['arm1_val']:.4f} {r['arm1_test']:.4f} | {r['arm3cf_val']:.4f} {r['arm3cf_test']:.4f} | "
          f"{r['margin_val']:+.4f} {r['margin_test']:+.4f} | {r['shift_full']:.4f}/{r['shift_img']:.4f} | "
          f"{r['arm2_val_agreement']:.3f}/{r['arm2_test_agreement']:.3f} | {r['leakage_rate']:.1%} | {r['minutes']:.0f}")

def summarize(name, xs):
    xs = np.array(xs, dtype=float)
    m, s = xs.mean(), xs.std(ddof=1) if len(xs) > 1 else float('nan')
    print(f'{name:36s} mean = {m:+.4f}   std = {s:.4f}   min/max = {xs.min():+.4f}/{xs.max():+.4f}')
    return m, s

print()
summarize('Arm 1 TEST AUROC', [r['arm1_test'] for r in all_results])
summarize('Arm 3c-frozen TEST AUROC', [r['arm3cf_test'] for r in all_results])
m_test, s_test = summarize('margin (3c-frozen - Arm 1), TEST', [r['margin_test'] for r in all_results])
m_val, s_val = summarize('margin (3c-frozen - Arm 1), val', [r['margin_val'] for r in all_results])
summarize('Arm 1 val AUROC (run-to-run spread)', [r['arm1_val'] for r in all_results])
summarize('Arm 3c-frozen val AUROC (spread)', [r['arm3cf_val'] for r in all_results])

margins = np.array([r['margin_test'] for r in all_results])
positive = int((margins > 0).sum())
print(f'\nseeds with a positive TEST margin: {positive}/{n}')
if n >= 2:
    t, p = stats.ttest_1samp(margins, 0.0)
    half = stats.t.ppf(0.975, n - 1) * margins.std(ddof=1) / np.sqrt(n)
    print(f'paired t-test of TEST margin vs 0: t = {t:.2f}, p = {p:.3f}   95% CI = [{m_test - half:+.4f}, {m_test + half:+.4f}]   (n = {n})')
    if m_test - half > 0:
        print('VERDICT: the 95% CI excludes zero -- H1 (fusion beats no-retrieval) holds on held-out test across seeds, with the CI above as the reportable magnitude.')
    elif positive == n:
        print('VERDICT: every seed favours fusion, but the CI includes zero at this n -- direction is consistent, magnitude is not yet established; more seeds would tighten it.')
    else:
        print('VERDICT: the margin changes sign across seeds -- H1 is not supported at a magnitude distinguishable from seed noise.')
else:
    print('(need >= 2 seeds for a CI -- add more seeds or paste PRIOR_RESULTS from another session)')

print('\nreference single-seed runs (SEED=42, from RESULTS.md; NOT in the statistics above):')
for name, ref in REFERENCE_RUNS.items():
    line = f"  {name:38s} Arm1 val {ref['arm1_val']:.4f}  Arm3cf val {ref['arm3cf_val']:.4f}  margin val {ref['arm3cf_val'] - ref['arm1_val']:+.4f}"
    if 'arm1_test' in ref:
        line += f"   | test {ref['arm1_test']:.4f} / {ref['arm3cf_test']:.4f}  margin test {ref['arm3cf_test'] - ref['arm1_test']:+.4f}"
    print(line)